# TimesFM 3 (modelo fundacional de Google, *zero-shot*): proceso

Cuaderno de proceso del modelo preentrenado de Google. Las celdas de código son lecturas ligeras
(no descargan pesos ni entrenan nada) y avisan si falta algún archivo. Modelos hermanos:
[`01_xgboost_booster.ipynb`](01_xgboost_booster.ipynb), [`02_tcn.ipynb`](02_tcn.ipynb),
[`03_cnn_lstm.ipynb`](03_cnn_lstm.ipynb) y [`04_transformer.ipynb`](04_transformer.ipynb).

> **Estado de la validación.** La búsqueda *zero-shot* (6 configuraciones) y la validación de 24
> cortes con `--folds` están terminadas, en CPU. La sección 6 compara con los demás modelos y la 7
> recoge las limitaciones, empezando por la posible contaminación del preentrenamiento.

In [ ]:
import os
import warnings
from pathlib import Path

import polars as pl
import yaml

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Raiz del repositorio:", ROOT)


def leer_csv(ruta):
    try:
        return pl.read_csv(ruta)
    except FileNotFoundError:
        print(f"No existe {ruta}. Ver la seccion 'Como reproducir'.")
    except Exception as error:
        print(f"No se pudo leer {ruta}: {error}")


def leer_yaml(ruta):
    with open(ruta, encoding="utf-8") as archivo:
        return yaml.safe_load(archivo)

## 1. Objetivo y criterio de selección

El mismo de los otros cuadernos: pronosticar el nivel en **Ciudad Bolívar** y **Palúa** a 1–14
días, reportando los **días 7 y 14**, con el criterio fijado antes de ver resultados (`rel_mae < 1`
en cada estación y, entre los que cumplen, el menor MAE medio).

**Qué pregunta responde este modelo.** Es el único de los seis que **no se entrena con datos del
Orinoco**: TimesFM ([Das et al., 2024](https://proceedings.mlr.press/v235/das24c.html)) se
preentrenó con miles de series y aquí solo recibe la historia hasta el corte como contexto.
Contesta si un modelo general supera a modelos ajustados para este río.

## 2. Datos y partición

Los mismos que el resto: hoja `Entrenar` de `DATOS RIO ORINOCO.xlsx`, **2009-01-01 a 2024-12-31**,
el frame de `load_history` con las cuatro estaciones, **24 cortes** de 2021-01-26 a 2022-12-17
cada 30 días y el **test (2023 en adelante) sin tocar**. Ver la sección 2 y 3 de
[`03_cnn_lstm.ipynb`](03_cnn_lstm.ipynb); no se repiten aquí.

## 3. API verificada

El README oficial de TimesFM documenta `TimesFM3Evaluator`, que **no existe** en `timesfm==3.0.2`
(verificado el 2026-10-29 instalando el extra). La API real:

```python
from timesfm import TimesFM3Forecaster
modelo = TimesFM3Forecaster.from_pretrained("google/timesfm-3.0-pytorch", device=None)
salidas = modelo.predict_batch(contextos, horizonte, past_only_covariates=[...])
# cada salida: .forecast (mediana) y .quantiles
```

`contextos` es una lista de arreglos 1D; `past_only_covariates`, uno `(k, contexto)` por serie. La
salida llega en parches de 64 pasos y el runner la recorta a 14. La celda siguiente lo comprueba
sin descargar pesos.

In [ ]:
try:
    import inspect

    import timesfm

    print("Clases expuestas:", [n for n in dir(timesfm) if n[:1].isupper()])
    print("predict_batch", inspect.signature(timesfm.TimesFM3Forecaster.predict_batch))
except ImportError:
    print("Falta el extra opcional: uv sync --extra timesfm")

## 4. Diseño del runner

Código: [`models/pretrained/timesfm.py`](../src/orinoco_forecast/models/pretrained/timesfm.py)
(`TimesFM3Runner`) y [`configs/models/timesfm.yaml`](../configs/models/timesfm.yaml).

| Decisión | Qué problema resuelve | Cómo funciona por dentro | Alternativa descartada y por qué | Cuándo sería un error |
|---|---|---|---|---|
| **Zero-shot** (sin ajuste fino) | Medir el modelo general, no uno adaptado | `fit` solo carga los pesos; `predict` pasa la historia hasta el corte | LoRA sobre los datos del río: rompe la pregunta "general contra específico" y no está documentado para la 3.0 | Si el objetivo fuera solo el menor error posible |
| **Pesos cargados una vez por proceso** | El backtest crea un runner por corte; cargar 24 veces los pesos sería lento | Caché por checkpoint y dispositivo (`load_forecaster`) | Cargar en cada `fit` | Si se cambiara de checkpoint dentro del mismo proceso sin vaciar la caché |
| **Contexto = últimos `context_len` días hasta el corte** | Que el modelo no vea nada posterior al corte | `y[ds <= corte][-context_len:]` por serie | Toda la historia: más lenta y, en la búsqueda, peor | Nunca mientras el corte se respete |
| **Covariables opcionales, solo pasadas** | Probar si los niveles aguas arriba ayudan | `past_only_covariates` con las columnas que deja `drop_covariates`, del mismo tramo | `past_future_covariates`: exigiría valores futuros de las estaciones, que no se conocen | Si se pasaran columnas con datos posteriores al corte |
| **Mismo `drop_covariates: [rain_, oni]`** que las redes | Comparar con los mismos insumos | Oculta lluvia y ONI | Pasarle todo | Si TimesFM aprovechara la lluvia (no se probó) |
| **Una sola columna `TIMESFM3`, sin semillas** | La inferencia es determinista | — | Varias semillas: no aportan nada | — |
| **Mediana como pronóstico puntual** | Comparar MAE con los demás | `.forecast` (cuantil 0,5) | Media de cuantiles | Si se evaluara un intervalo, no un punto |

**Licencia.** Los pesos `google/timesfm-3.0-pytorch` son de uso **no comercial / no productivo**;
compatible con un proyecto académico. El código es Apache-2.0.

In [ ]:
try:
    cfg = leer_yaml("configs/models/timesfm.yaml")
    for clave in [
        "checkpoint",
        "device",
        "context_len",
        "use_covariates",
        "drop_covariates",
        "use_symmetric_averaging",
    ]:
        print(f"{clave:24s} {cfg.get(clave)}")
except FileNotFoundError:
    print("No existe configs/models/timesfm.yaml")

## 5. Búsqueda *zero-shot* (en lugar de 20 pruebas de Optuna)

TimesFM no se entrena, así que su espacio es **finito**: `context_len` ∈ {512, 1024, 2048} ×
`use_covariates` ∈ {no, sí} = **6 configuraciones, todas evaluadas**
([`scripts/tune_timesfm3.py`](../scripts/tune_timesfm3.py)). Mismos 12 cortes de ajuste de 2020 y
mismo objetivo (`rel_mae` medio) que los demás. Que sean 6 y no 20 no es un presupuesto menor: el
espacio queda agotado.

| Decisión | Qué problema resuelve | Cómo funciona | Alternativa descartada | Cuándo sería un error |
|---|---|---|---|---|
| **Rejilla exhaustiva** | Espacio pequeño y discreto | Un bucle sobre las 6 combinaciones | TPE con 20 pruebas: repetiría combinaciones | Si el espacio fuera continuo o grande |
| **Naive reajustado en cada corte** | `NaiveRunner.predict` ignora el `cutoff` y repite el nivel del corte en que se ajustó | Un ajuste de Naive por corte | Ajustarlo una vez, como `tune_cnn_lstm.py`: el denominador de `rel_mae` saldría solo del primer corte | — |

**Por eso los `rel_mae` de esta búsqueda no se comparan con los de los otros CSV de ajuste**: en
aquellos el denominador es la persistencia del primer corte de ajuste (no cambia el orden de sus
pruebas, pero sí el valor). La comparación válida es la de 24 cortes (sección 6).

**Resultado:** gana `context_len` 1024 **sin covariables** (`rel_mae` 0,454: Ciudad Bolívar 0,35,
Palúa 0,56), que eran los valores iniciales. Las covariables mejoran Palúa (hasta 0,48) pero
empeoran Ciudad Bolívar más de lo que ganan.

In [ ]:
pruebas = leer_csv("reports/tuning/timesfm3_trials.csv")
if pruebas is not None:
    with pl.Config(tbl_cols=-1, tbl_width_chars=200, float_precision=3):
        print(
            pruebas.sort("value").select(
                "number",
                "value",
                "rel_mae_ciudad_bolivar",
                "rel_mae_palua",
                "params_context_len",
                "params_use_covariates",
                "duration_s",
            )
        )

## 6. Validación de 24 cortes (la que cuenta para elegir)

TimesFM 3 en CPU contra Naive (`MAE / rel_mae`; `p` = test de signos contra Naive):

| Estación | Horizonte | Naive | TimesFM 3 | p |
|---|---|---|---|---|
| Ciudad Bolívar | día 7 | 0,526 | 0,278 / 0,53 | 0,007 |
| Ciudad Bolívar | día 14 | 1,064 | 0,507 / 0,48 | 0,007 |
| Ciudad Bolívar | días 1 a 14 | 0,570 | 0,287 / 0,50 | 0,002 |
| Palúa | día 7 | 0,435 | 0,345 / 0,79 | 0,152 |
| Palúa | día 14 | 0,891 | 0,486 / 0,55 | 0,023 |
| Palúa | días 1 a 14 | 0,488 | 0,313 / 0,64 | 0,064 |

MAE medio entre estaciones (días 1 a 14): **TimesFM 3 0,300 m**, XGBoost 0,309, TCN 0,317,
CNN-LSTM 0,350 y ARIMAX 0,351 (el Transformer está en su cuaderno).

**Lectura.**

- **Cumple el criterio** en las dos estaciones y los tres horizontes y tiene el **menor MAE medio**
  de los modelos validados, sin haber visto un solo dato del Orinoco en su entrenamiento.
- **Palúa es donde gana**: mejor MAE a 7 y a 14 días de todos los modelos. A 7 días gana a Naive en
  el 67 % de los cortes (p = 0,152, sin evidencia todavía; el booster y el TCN tienen p ≥ 0,54).
- **Ciudad Bolívar**: a la par del XGBoost y del TCN (0,287 frente a 0,278 y 0,292).
- La diferencia con el XGBoost (0,009 m) es pequeña frente a la dispersión entre cortes; ver la
  celda siguiente antes de declararlo ganador.

In [ ]:
def mae_por_modelo(ruta, nombre):
    try:
        tabla = pl.read_parquet(ruta)
    except FileNotFoundError:
        print(f"No existe {ruta}")
        return None
    return (
        tabla.filter(
            (pl.col("metric") == "mae")
            & pl.col("horizon").cast(pl.String).is_in(["7", "14", "all"])
        )
        .group_by("unique_id", "horizon")
        .agg(
            pl.lit(nombre).alias("modelo"),
            pl.col("mean_model").mean().alias("mae"),
            pl.col("std_model").mean().alias("desv_cortes"),
            pl.col("sign_test_p").min().alias("p"),
        )
    )


fuentes = {
    "timesfm3": "reports/predictions/timesfm3/backtest/naive/comparison.parquet",
    "booster": "reports/predictions/booster/backtest/naive/comparison.parquet",
    "tcn": "reports/predictions/tcn/backtest/naive/comparison.parquet",
    "cnn_lstm": "reports/predictions/cnn_lstm/backtest/naive_gpu/comparison.parquet",
    "arimax": "reports/predictions/arimax/backtest/naive/comparison.parquet",
    "transformer": "reports/predictions/transformer/backtest/naive/comparison.parquet",
}
partes = [t for n, r in fuentes.items() if (t := mae_por_modelo(r, n)) is not None]
if partes:
    with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=200, float_precision=3):
        print(pl.concat(partes).sort("unique_id", "horizon", "mae"))

## 7. Limitaciones

- **Posible contaminación del preentrenamiento.** TimesFM se entrenó con miles de series públicas.
  Si entre ellas había niveles o caudales del Orinoco de 2021–2022, esta validación sería
  optimista. **No se puede verificar**; es la limitación principal de este modelo.
- **CPU.** Los 24 cortes corrieron en CPU (53 s); TimesFM es determinista en CPU, pero en GPU
  podría diferir levemente.
- **Sin cuantiles en la comparación.** El modelo entrega cuantiles 0,1–0,9; se evaluó solo la
  mediana para comparar MAE con el resto.
- **Búsqueda con 6 configuraciones** y otra definición de Naive que la de los otros CSV de ajuste
  (sección 5).
- **Sin lluvia ni ONI** (`drop_covariates`), igual que las redes; no se probó con la lluvia de las
  subcuencas (`feat/basin-rainfall`).
- **Test final sin tocar.**

## 8. Cómo reproducir

```bash
uv sync --extra timesfm
# Genera el parquet (una vez)
uv run python scripts/run_model.py booster --write-dataset
# Busqueda zero-shot (6 configuraciones, unos 7 min en CPU)
uv run python scripts/tune_timesfm3.py
# Un corte contra Naive (descarga los pesos la primera vez)
uv run python scripts/run_model.py timesfm3 --against naive
# 24 cortes (salidas en data/predictions/timesfm3/backtest/naive/)
uv run python scripts/run_model.py timesfm3 --against naive --folds
```

En Colab: [`colab_folds.ipynb`](colab_folds.ipynb) con `MODEL = "timesfm3"` (instala el extra).